--------

We do this after an extensive testing phase in stat_testing. This is done at the end because stat_testing checks different metrics of IC features. So once we have completed the check we can confidently saying that these features actaully add value. Then stationarity testing checks if these signals persist and are reliable

stats_testing is on IC time series while the stat_stationary is on the feature itself

--------

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
from tqdm import tqdm

In [2]:
features_df = pd.read_parquet("/Users/sudhanvabharadwaj/Documents/Full_Quant_Pipeline/Data/processed/splits/train.parquet")
features_df.head()

,date,ticker,fwd_return_5d,dollar_volume,price_ma_ratio_5,price_ma_ratio_10,price_ma_ratio_21,price_ma_ratio_63,price_ma_ratio_252,pos_days_21,...,downside_vol_1d_rank,downside_vol_5d_rank,vol_avg_5_rank,vol_avg_21_rank,vol_avg_63_rank,dollar_vol_log_rank,volume_trend_rank,mom_x_vol_5_rank,mom_x_vol_21_rank,mom_x_vol_63_rank
0,2011-01-03,AAPL,0.039081,4.399815e+09,0.013095,0.015186,0.022646,0.053737,0.266111,0.571429,...,0.078947,0.105263,0.947368,0.947368,0.973684,1.000000,0.078947,0.947368,0.947368,0.973684
1,2011-01-04,AAPL,0.031242,3.070944e+09,0.014752,0.017638,0.025883,0.056965,0.270454,0.619048,...,0.052632,0.105263,0.947368,0.947368,0.973684,1.000000,0.078947,0.947368,0.947368,0.947368
2,2011-01-05,AAPL,0.031198,2.559542e+09,0.017622,0.022883,0.032167,0.063198,0.278453,0.619048,...,0.078947,0.078947,0.947368,0.947368,0.973684,0.973684,0.078947,0.947368,0.947368,0.947368
3,2011-01-06,AAPL,0.035807,3.006965e+09,0.010599,0.019381,0.028983,0.059955,0.275035,0.619048,...,0.052632,0.052632,0.947368,0.947368,0.973684,1.000000,0.052632,0.947368,0.947368,0.947368
4,2011-01-07,AAPL,0.036773,3.144450e+09,0.009545,0.022770,0.034058,0.065287,0.281753,0.619048,...,0.052632,0.052632,0.947368,0.947368,0.973684,0.973684,0.052632,0.947368,0.947368,0.947368


------------

We need to check stationarity per time series, but this is way too computationally expensive so we use 2 different approaches

1. Panel proxy test 

This tests cross-sectional median time series. feature_t = median(feature over all stocks at time t). Tells us if feature drifting overall

---------

In [3]:
df = features_df
df

,date,ticker,fwd_return_5d,dollar_volume,price_ma_ratio_5,price_ma_ratio_10,price_ma_ratio_21,price_ma_ratio_63,price_ma_ratio_252,pos_days_21,...,downside_vol_1d_rank,downside_vol_5d_rank,vol_avg_5_rank,vol_avg_21_rank,vol_avg_63_rank,dollar_vol_log_rank,volume_trend_rank,mom_x_vol_5_rank,mom_x_vol_21_rank,mom_x_vol_63_rank
0,2011-01-03,AAPL,0.039081,4.399815e+09,0.013095,0.015186,0.022646,0.053737,0.266111,0.571429,...,0.078947,0.105263,0.947368,0.947368,0.973684,1.000000,0.078947,0.947368,0.947368,0.973684
1,2011-01-04,AAPL,0.031242,3.070944e+09,0.014752,0.017638,0.025883,0.056965,0.270454,0.619048,...,0.052632,0.105263,0.947368,0.947368,0.973684,1.000000,0.078947,0.947368,0.947368,0.947368
2,2011-01-05,AAPL,0.031198,2.559542e+09,0.017622,0.022883,0.032167,0.063198,0.278453,0.619048,...,0.078947,0.078947,0.947368,0.947368,0.973684,0.973684,0.078947,0.947368,0.947368,0.947368
3,2011-01-06,AAPL,0.035807,3.006965e+09,0.010599,0.019381,0.028983,0.059955,0.275035,0.619048,...,0.052632,0.052632,0.947368,0.947368,0.973684,1.000000,0.052632,0.947368,0.947368,0.947368
4,2011-01-07,AAPL,0.036773,3.144450e+09,0.009545,0.022770,0.034058,0.065287,0.281753,0.619048,...,0.052632,0.052632,0.947368,0.947368,0.973684,0.973684,0.052632,0.947368,0.947368,0.947368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109279,2021-12-27,XOM,0.026660,6.762030e+08,0.019403,0.015589,0.010677,-0.002264,0.102931,0.571429,...,0.625000,0.575000,0.625000,0.650000,0.675000,0.400000,0.225000,0.700000,0.325000,0.625000
109280,2021-12-28,XOM,0.068731,6.840421e+08,0.007710,0.012208,0.007066,-0.006141,0.097630,0.571429,...,0.525000,0.550000,0.625000,0.625000,0.675000,0.425000,0.350000,0.700000,0.425000,0.500000
109281,2021-12-29,XOM,0.091578,6.755398e+08,-0.003227,0.003990,-0.001407,-0.015360,0.086314,0.523810,...,0.550000,0.550000,0.625000,0.650000,0.675000,0.425000,0.250000,0.475000,0.275000,0.500000
109282,2021-12-30,XOM,0.123869,6.297245e+08,-0.008449,-0.001134,-0.008019,-0.021845,0.078277,0.523810,...,0.375000,0.575000,0.625000,0.625000,0.650000,0.400000,0.275000,0.125000,0.450000,0.575000


In [4]:
df["date"] = pd.to_datetime(df["date"])

In [5]:
rank_cols = [col for col in df.columns if '_rank' in col]
df = df.drop(columns=rank_cols)

df

,date,ticker,fwd_return_5d,dollar_volume,price_ma_ratio_5,price_ma_ratio_10,price_ma_ratio_21,price_ma_ratio_63,price_ma_ratio_252,pos_days_21,...,downside_vol_1d,downside_vol_5d,vol_avg_5,vol_avg_21,vol_avg_63,dollar_vol_log,volume_trend,mom_x_vol_5,mom_x_vol_21,mom_x_vol_63
0,2011-01-03,AAPL,0.039081,4.399815e+09,0.013095,0.015186,0.022646,0.053737,0.266111,0.571429,...,0.002118,0.001247,227040800.0,3.092279e+08,4.488507e+08,22.204828,-8.312502e+06,3.419588e+06,1.109984e+07,8.204138e+07
1,2011-01-04,AAPL,0.031242,3.070944e+09,0.014752,0.017638,0.025883,0.056965,0.270454,0.619048,...,0.002138,0.001247,253672160.0,3.076559e+08,4.457890e+08,21.845251,-7.844913e+06,4.536055e+06,1.342301e+07,6.533927e+07
2,2011-01-05,AAPL,0.031198,2.559542e+09,0.017622,0.022883,0.032167,0.063198,0.278453,0.619048,...,0.002138,0.001247,272148240.0,2.984672e+08,4.391961e+08,21.663094,-6.510211e+06,7.287014e+06,1.291194e+07,6.805336e+07
3,2011-01-06,AAPL,0.035807,3.006965e+09,0.010599,0.019381,0.028983,0.059955,0.275035,0.619048,...,0.001858,0.001247,300735120.0,2.941327e+08,4.374823e+08,21.824197,-5.156353e+06,9.356793e+06,1.434567e+07,6.732704e+07
4,2011-01-07,AAPL,0.036773,3.144450e+09,0.009545,0.022770,0.034058,0.065287,0.281753,0.619048,...,0.001858,0.001247,324419760.0,2.936563e+08,4.319828e+08,21.868905,-4.528018e+06,1.363817e+07,1.382247e+07,6.177061e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109279,2021-12-27,XOM,0.026660,6.762030e+08,0.019403,0.015589,0.010677,-0.002264,0.102931,0.571429,...,0.010966,0.017684,16362700.0,2.192815e+07,2.103481e+07,20.332004,-4.292123e+05,5.069886e+05,-5.492418e+05,1.213762e+06
109280,2021-12-28,XOM,0.068731,6.840421e+08,0.007710,0.012208,0.007066,-0.006141,0.097630,0.571429,...,0.008782,0.017253,14555040.0,2.134727e+07,2.064055e+07,20.343530,-5.044144e+05,6.224543e+05,1.533547e+05,8.952968e+05
109281,2021-12-29,XOM,0.091578,6.755398e+08,-0.003227,0.003990,-0.001407,-0.015360,0.086314,0.523810,...,0.008745,0.017253,13141980.0,2.105142e+07,2.047088e+07,20.331023,-6.505783e+05,1.411958e+05,-1.503915e+05,7.151160e+05
109282,2021-12-30,XOM,0.123869,6.297245e+08,-0.008449,-0.001134,-0.008019,-0.021845,0.078277,0.523810,...,0.006973,0.017101,12718380.0,1.995430e+07,2.016375e+07,20.260793,-5.699277e+05,-4.170689e+04,3.167861e+05,9.553846e+05


In [6]:
cols = ["date", "ticker", "fwd_return_5d"]

feature_cols = [c for c in df.columns if c not in cols]

len(feature_cols)

34

In [7]:
#Creating the cross-sectional median time series

cs_median = (df.groupby('date')[feature_cols].median().sort_index())
cs_median

,dollar_volume,price_ma_ratio_5,price_ma_ratio_10,price_ma_ratio_21,price_ma_ratio_63,price_ma_ratio_252,pos_days_21,z_price_21,z_price_63,z_return_1d_5,...,downside_vol_1d,downside_vol_5d,vol_avg_5,vol_avg_21,vol_avg_63,dollar_vol_log,volume_trend,mom_x_vol_5,mom_x_vol_21,mom_x_vol_63
date,,,,,,,,,,,,,,,,,,,,,
2011-01-03,3.982696e+08,0.007083,0.008197,0.017294,0.061871,0.128499,0.571429,1.234197,1.279883,2.239095,...,0.003409,0.004987,8294500.0,1.146872e+07,1.396176e+07,19.802640,-357718.961039,57985.853077,344374.431804,2.085300e+06
2011-01-04,4.398899e+08,0.008990,0.009354,0.020620,0.063554,0.128040,0.571429,1.293668,1.267026,0.029923,...,0.003554,0.004987,10625060.0,1.162309e+07,1.401593e+07,19.902035,-321437.272727,98763.986334,349191.304662,1.734041e+06
2011-01-05,5.489701e+08,0.007743,0.011471,0.019395,0.057782,0.131290,0.619048,1.130567,1.298742,-0.024432,...,0.004107,0.005197,13489080.0,1.181553e+07,1.413203e+07,20.123555,-266854.155844,125968.971105,479591.115746,2.112840e+06
2011-01-06,5.027351e+08,0.007741,0.012863,0.023421,0.059316,0.129876,0.619048,1.397361,1.367528,-0.217675,...,0.004111,0.005663,15319500.0,1.176967e+07,1.410030e+07,20.035574,-145525.844156,211869.208733,375225.030867,1.354964e+06
2011-01-07,3.634272e+08,0.002313,0.009454,0.016050,0.058746,0.126862,0.571429,1.128481,1.246488,-0.350618,...,0.004465,0.005663,17076860.0,1.197115e+07,1.416805e+07,19.711090,-124141.428571,282345.659025,392688.277869,1.428127e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,8.209561e+08,0.018583,0.018210,0.030422,0.061221,0.160400,0.523810,1.207871,1.295107,0.892748,...,0.009442,0.017412,9629550.0,1.300879e+07,1.087416e+07,20.525943,-128164.025974,223425.774717,134980.761594,7.707212e+05
2021-12-28,7.766628e+08,0.010416,0.017660,0.028770,0.058084,0.161585,0.523810,1.127837,1.244015,-0.497232,...,0.008663,0.016255,8652640.0,1.283960e+07,1.074645e+07,20.468038,-205363.896104,265181.077735,397432.395645,8.954159e+05
2021-12-29,7.648280e+08,0.007241,0.016496,0.029198,0.062429,0.160852,0.523810,1.224417,1.044217,-0.698422,...,0.008442,0.016334,7411950.0,1.240627e+07,1.070924e+07,20.454782,-269159.545455,164788.152783,265795.084696,7.506046e+05


In [8]:
def run_stationarity_tests(series):
    
    series = series.dropna()
    
    adf_stat, adf_p, *_ = adfuller(series, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    
    return adf_stat, adf_p, kpss_stat, kpss_p

In [9]:
results = []

for feature in tqdm(feature_cols):
    
    try:
        
        adf_stat, adf_p, kpss_stat, kpss_p = run_stationarity_tests(cs_median[feature])
        results.append({
            "feature": feature,
            "adf_stat": adf_stat,
            "adf_p": adf_p,
            "kpss_stat": kpss_stat,
            "kpss_p": kpss_p
        })
        
    except:
        continue

stationarity_df = pd.DataFrame(results)
stationarity_df.head()

  0%|          | 0/34 [00:00<?, ?it/s]/var/folders/gx/p5qhhh8s7rg2_ywh443qwth80000gn/T/ipykernel_69357/2478106527.py:6: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
  3%|▎         | 1/34 [00:00<00:05,  5.62it/s]/var/folders/gx/p5qhhh8s7rg2_ywh443qwth80000gn/T/ipykernel_69357/2478106527.py:6: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
/var/folders/gx/p5qhhh8s7rg2_ywh443qwth80000gn/T/ipykernel_69357/2478106527.py:6: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(

,feature,adf_stat,adf_p,kpss_stat,kpss_p
0,dollar_volume,-2.387761,1.452207e-01,6.702140,0.01
1,price_ma_ratio_5,-11.388897,8.159925e-21,0.070425,0.10
2,price_ma_ratio_10,-11.027649,5.773766e-20,0.067270,0.10
3,price_ma_ratio_21,-12.203542,1.207922e-22,0.050580,0.10
4,price_ma_ratio_63,-7.944841,3.240696e-12,0.141597,0.10


In [10]:
stationarity_df.shape

(34, 5)

In [11]:
def classify_stationarity(row):
    
    adf_stationary = row['adf_p'] < 0.05
    kpss_stationary = row['kpss_p'] > 0.05

    if adf_stationary and kpss_stationary:
        return "stationary"
    elif not adf_stationary and not kpss_stationary:
        return "non_stationary"
    else:
        return "borderline"

stationarity_df["regime"] = stationarity_df.apply(classify_stationarity, axis=1)
stationarity_df["regime"].value_counts()

regime
stationary        22
borderline        10
non_stationary     2
Name: count, dtype: int64

In [12]:
stationarity_df.sort_values("adf_p").head(10)

,feature,adf_stat,adf_p,kpss_stat,kpss_p,regime
9,z_return_1d_5,-21.161749,0.000000e+00,0.119298,0.1,stationary
10,z_return_1d_21,-14.755813,2.444942e-27,0.015944,0.1,stationary
13,z_return_5d_21,-14.412329,8.165603e-27,0.009100,0.1,stationary
12,z_return_5d_5,-14.395187,8.691391e-27,0.020636,0.1,stationary
14,z_return_5d_35,-13.583818,2.099036e-25,0.008090,0.1,stationary
11,z_return_1d_35,-13.131342,1.492557e-24,0.011420,0.1,stationary
22,vol_ratio_5_21,-12.943402,3.496184e-24,0.043169,0.1,stationary
15,rev_1d,-12.835981,5.739576e-24,0.067617,0.1,stationary
31,mom_x_vol_5,-12.470679,3.250115e-23,0.016705,0.1,stationary
3,price_ma_ratio_21,-12.203542,1.207922e-22,0.050580,0.1,stationary


In [13]:
stationarity_df.sort_values("kpss_p").head(10)

,feature,adf_stat,adf_p,kpss_stat,kpss_p,regime
0,dollar_volume,-2.387761,1.452207e-01,6.702140,0.010000,non_stationary
27,vol_avg_21,-4.740508,7.058093e-05,2.542923,0.010000,borderline
28,vol_avg_63,-3.280317,1.578156e-02,3.476765,0.010000,borderline
26,vol_avg_5,-4.958038,2.685227e-05,2.372537,0.010000,borderline
29,dollar_vol_log,-2.267720,1.825905e-01,7.435238,0.010000,non_stationary
20,vol_63,-4.697406,8.510194e-05,0.735724,0.010298,borderline
6,pos_days_21,-8.122972,1.142887e-12,0.654234,0.017706,borderline
5,price_ma_ratio_252,-4.362053,3.462559e-04,0.616886,0.021101,borderline
23,hl_range,-7.394937,7.822998e-11,0.551873,0.029984,borderline
19,vol_21,-5.620901,1.145767e-06,0.516125,0.038035,borderline


--------

2. Subsample testing(randomly choose stocks and test) 

-------

In [14]:
tickers = df['ticker'].unique()
sample_tickers = np.random.choice(tickers, 30, replace=False)

Find feature values where the regime is not stationary and then make a dataframe where you have each stock and the coresspoding value of that feature.

In [15]:
test_feat = stationarity_df.query("regime=='non_stationary'").iloc[0]["feature"]

bad_series = (
    df[df["ticker"].isin(sample_tickers)]
    .pivot(index="date", columns="ticker", values=test_feat)
)

bad_series.head()

ticker,ABBV,ABT,ADBE,AMD,AMZN,AVGO,BAC,COST,CSCO,CVX,...,MSFT,NFLX,NKE,ORCL,PFE,PG,QCOM,SBUX,TMO,TSLA
date,,,,,,,,,,,,,,,,,,,,,
2011-01-03,NaN,3.168167e+08,1.954217e+08,1.791651e+08,9.821506e+08,2.834557e+07,3.911203e+09,1.772581e+08,7.200493e+08,3.921939e+08,...,1.140810e+09,1.018364e+09,1.519747e+08,5.383669e+08,3.052901e+08,4.168346e+08,5.182173e+08,1.606543e+08,1.292017e+08,NaN
2011-01-04,NaN,3.160406e+08,2.532805e+08,2.356964e+08,9.309333e+08,3.216070e+07,2.425718e+09,2.060701e+08,6.144001e+08,4.263809e+08,...,1.165906e+09,1.141742e+09,2.387525e+08,5.845556e+08,4.038776e+08,4.735304e+08,8.443418e+08,1.635901e+08,1.670051e+08,NaN
2011-01-05,NaN,4.757910e+08,2.301636e+08,1.733654e+08,6.407515e+08,1.885420e+07,2.776510e+09,2.177220e+08,9.324207e+08,3.329291e+08,...,1.260285e+09,8.164773e+08,2.020192e+08,9.158231e+08,6.271707e+08,3.109616e+08,9.176748e+08,1.408424e+08,1.426669e+08,NaN
2011-01-06,NaN,5.439652e+08,2.010711e+08,1.842123e+08,5.909791e+08,1.933409e+07,2.714555e+09,1.664586e+08,8.925826e+08,3.084463e+08,...,1.935418e+09,8.134498e+08,1.390724e+08,5.532034e+08,1.133909e+09,4.821179e+08,6.417109e+08,1.603347e+08,1.261918e+08,NaN
2011-01-07,NaN,3.634272e+08,2.187851e+08,1.241710e+08,9.685731e+08,1.752550e+07,4.349051e+09,1.188765e+08,9.190354e+08,3.164038e+08,...,1.609411e+09,5.674846e+08,1.407080e+08,6.978118e+08,1.279651e+09,4.954275e+08,4.531381e+08,2.455721e+08,8.540818e+07,NaN


In [16]:
test_feat

'dollar_volume'

In [17]:
bad_series.shape

(2769, 30)

In [18]:
bad_series

ticker,ABBV,ABT,ADBE,AMD,AMZN,AVGO,BAC,COST,CSCO,CVX,...,MSFT,NFLX,NKE,ORCL,PFE,PG,QCOM,SBUX,TMO,TSLA
date,,,,,,,,,,,,,,,,,,,,,
2011-01-03,NaN,3.168167e+08,1.954217e+08,1.791651e+08,9.821506e+08,2.834557e+07,3.911203e+09,1.772581e+08,7.200493e+08,3.921939e+08,...,1.140810e+09,1.018364e+09,1.519747e+08,5.383669e+08,3.052901e+08,4.168346e+08,5.182173e+08,1.606543e+08,1.292017e+08,NaN
2011-01-04,NaN,3.160406e+08,2.532805e+08,2.356964e+08,9.309333e+08,3.216070e+07,2.425718e+09,2.060701e+08,6.144001e+08,4.263809e+08,...,1.165906e+09,1.141742e+09,2.387525e+08,5.845556e+08,4.038776e+08,4.735304e+08,8.443418e+08,1.635901e+08,1.670051e+08,NaN
2011-01-05,NaN,4.757910e+08,2.301636e+08,1.733654e+08,6.407515e+08,1.885420e+07,2.776510e+09,2.177220e+08,9.324207e+08,3.329291e+08,...,1.260285e+09,8.164773e+08,2.020192e+08,9.158231e+08,6.271707e+08,3.109616e+08,9.176748e+08,1.408424e+08,1.426669e+08,NaN
2011-01-06,NaN,5.439652e+08,2.010711e+08,1.842123e+08,5.909791e+08,1.933409e+07,2.714555e+09,1.664586e+08,8.925826e+08,3.084463e+08,...,1.935418e+09,8.134498e+08,1.390724e+08,5.532034e+08,1.133909e+09,4.821179e+08,6.417109e+08,1.603347e+08,1.261918e+08,NaN
2011-01-07,NaN,3.634272e+08,2.187851e+08,1.241710e+08,9.685731e+08,1.752550e+07,4.349051e+09,1.188765e+08,9.190354e+08,3.164038e+08,...,1.609411e+09,5.674846e+08,1.407080e+08,6.978118e+08,1.279651e+09,4.954275e+08,4.531381e+08,2.455721e+08,8.540818e+07,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,5.101574e+08,4.964255e+08,1.143864e+09,8.226832e+09,9.957563e+09,1.214347e+09,1.366117e+09,1.213993e+09,8.280765e+08,7.829344e+08,...,6.609415e+09,1.263947e+09,5.926430e+08,6.484176e+08,1.297200e+09,6.617557e+08,8.290945e+08,3.855839e+08,3.480045e+08,2.594312e+10
2021-12-28,4.871344e+08,4.273791e+08,1.200097e+09,8.989767e+09,9.324575e+09,1.068811e+09,1.311957e+09,6.209172e+08,7.220501e+08,6.573783e+08,...,5.171235e+09,1.149845e+09,4.151153e+08,4.698270e+08,1.714262e+09,8.672757e+08,9.125413e+08,4.788303e+08,5.302981e+08,2.188695e+10
2021-12-29,6.477970e+08,3.712989e+08,1.414230e+09,7.605767e+09,6.049613e+09,9.219313e+08,1.019742e+09,9.476205e+08,8.515251e+08,6.982982e+08,...,4.976872e+09,7.858871e+08,6.066416e+08,4.689949e+08,1.206720e+09,7.884408e+08,9.223859e+08,4.455805e+08,4.221408e+08,2.033130e+10


We checked median but now we should know whether for some tickers, is it actually stationary

In [19]:
pvals = []

for col in bad_series:
    series = bad_series[col].dropna()
    if len(series) > 200:
        pvals.append(adfuller(series)[1])

np.mean(np.array(pvals) < 0.05)

np.float64(0.7333333333333333)

Meaning most assets are individually stationary for this feature, but the cross-sectional behavior is unstable. 
This is because:
Some assets have strong trends or structural breaks or regime shifts and they dominate the cross-sectional behavior
So the overall system behaves non-stationary, even though most individual parts do not.

This should be done for one of each kind of feature(one momentum, one MR etc)

Interpretation of results:

Non-stationary cross sectionally and majority stationary(80-85 percent) is a good case. Too much non-stationarity or extreme stationarity is also not great.

Cross-section: dynamic, rotating, non-stationary
Time-series: stable, mean-reverting, stationary

--------

Ways to fix:
    
1. Borderline Stationary: Transform, Regularize, difference etc

2. Non-stationary: Always transform, drop if still same problem

For stationary per stock majorly but less cross sectional movement, can combine features to make it stronger. Worst case drop them

------------